In [1]:
# Merge college production with combine and draft AV

import pandas as pd
import numpy as np
from pathlib import Path

# Directories
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
combine_av_df = pd.read_csv(RAW_DIR / "combine_vs_av_2000_2026.csv")

print(combine_av_df.shape)
print(combine_av_df.columns)
print(combine_av_df.describe())

(6644, 26)
Index(['season', 'round', 'pick', 'team', 'hof', 'position', 'category',
       'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started',
       'dr_av', 'player_name', 'pos', 'school', 'ht', 'wt', 'forty', 'bench',
       'vertical', 'broad_jump', 'cone', 'shuttle'],
      dtype='str')
            season        round         pick          age           to  \
count  6644.000000  6644.000000  6644.000000  6380.000000  6059.000000   
mean   2012.526490     4.497441   128.290036    22.511285  2017.128899   
std       7.510325     2.295266    73.808551     0.931263     7.072566   
min    2000.000000     1.000000     1.000000    20.000000  2000.000000   
25%    2006.000000     2.000000    64.000000    22.000000  2011.000000   
50%    2013.000000     4.000000   128.000000    23.000000  2018.000000   
75%    2019.000000     6.000000   192.000000    23.000000  2025.000000   
max    2025.000000     9.000000   262.000000    29.000000  2025.000000   

            allpro 

In [3]:
college_df = pd.read_csv(PROCESSED_DIR / "cfb_stats_with_avg_sp_and_av.csv")

print(college_df.shape)
print(college_df.columns)
print(college_df.describe())

(4844, 60)
Index(['Player', 'Rate', 'Draft Team', 'Round', 'Pick', 'Draft Year',
       'Draft College', 'From', 'To', 'G', 'qb_Cmp', 'qb_Att', 'qb_Inc',
       'qb_Cmp%', 'qb_Yds', 'qb_TD', 'qb_Int', 'qb_TD%', 'qb_Int%', 'qb_Y/A',
       'qb_AY/A', 'qb_Y/C', 'qb_Y/G', 'Pos', 'Team', 'rb_rush_Att',
       'rb_rush_Yds', 'rb_rush_Y_per_Att', 'rb_rush_TD', 'rb_rec_Rec',
       'rb_rec_Yds', 'rb_rec_Y_per_Rec', 'rb_rec_TD', 'rb_rush_Y_per_Game',
       'rb_rec_Y_per_Game', 'rec_Y/G', 'rec_Rec', 'rec_Yds', 'rec_Y/R',
       'rec_TD', 'Sk', 'Solo', 'Ast', 'Comb', 'TFL', 'Sk/G', 'Solo/G', 'Ast/G',
       'Comb/G', 'TFL/G', 'Int', 'db_ret_Yds', 'IntTD', 'PD', 'Int/G', 'PD/G',
       'avg_team_sp', 'age', 'dr_av', 'w_av'],
      dtype='str')
             Rate        Round         Pick   Draft Year         From  \
count  274.000000  4844.000000  4844.000000  4844.000000  4844.000000   
mean   135.249635     3.994013   120.996078  2011.954789  2008.658547   
std     33.468414     2.005301    73.

In [4]:
college_df.head()

,Player,Rate,Draft Team,Round,Pick,Draft Year,Draft College,From,To,G,...,Int,db_ret_Yds,IntTD,PD,Int/G,PD/G,avg_team_sp,age,dr_av,w_av
0,Courtney Brown,NaN,CLE,1,1,2000,Penn State,1999,1999,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,21.0,27.0
1,Lavar Arrington,NaN,WAS,1,2,2000,Penn State,1998,1999,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,45.0,46.0
2,Chris Samuels,NaN,WAS,1,3,2000,Alabama,1999,1999,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,63.0,63.0
3,Peter Warrick,NaN,CIN,1,4,2000,Florida State,1995,1999,54.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,25.0,27.0
4,Jamal Lewis,NaN,RAV,1,5,2000,Tennessee,1997,1999,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,53.0,69.0


In [5]:
combine_av_df = combine_av_df.rename(columns={
    "player_name": "player",
    "season": "draft_year"
})

college_df = college_df.rename(columns={
    "Draft Year": "draft_year",
    "Pick": "pick",
    "Round": "round",
    "Player": "player",
    "Pos": "pos"
})

In [6]:
college_df = college_df.dropna(subset=["draft_year", "pick"])

In [7]:
college_df["draft_year"] = college_df["draft_year"].astype(int)
college_df["pick"] = college_df["pick"].astype(int)

In [10]:
print(college_df[["draft_year", "pick", "round"]].dtypes)

draft_year    int64
pick          int64
round         int64
dtype: object


In [8]:
dup_count = college_df.duplicated(subset=["draft_year", "pick"]).sum()
print("Duplicate (year, pick):", dup_count)

Duplicate (year, pick): 273


In [9]:
college_df = (
    college_df
    .sort_values(by="G", ascending=False)  # keep most complete season
    .drop_duplicates(subset=["draft_year", "pick"], keep="first")
)

dup_count = college_df.duplicated(subset=["draft_year", "pick"]).sum()
print("Duplicate (year, pick):", dup_count)

Duplicate (year, pick): 0


In [11]:
college_cols = [
    # Keys
    "draft_year", "pick", "round",

    # Metadata
    "pos", "G", "age", "avg_team_sp",

    # QB
    "qb_Cmp", "qb_Att", "qb_Cmp%", "qb_Yds", "qb_TD", "qb_Int",
    "qb_Y/A", "qb_AY/A", "qb_Y/G",

    # RB
    "rb_rush_Att", "rb_rush_Yds", "rb_rush_Y_per_Att", "rb_rush_TD",
    "rb_rec_Rec", "rb_rec_Yds", "rb_rec_Y_per_Rec", "rb_rec_TD",

    # WR
    "rec_Rec", "rec_Yds", "rec_Y/R", "rec_TD", "rec_Y/G",

    # Defense
    "Sk", "Solo", "Ast", "Comb", "TFL",
    "Int", "PD"
]

# Keep only columns that actually exist
college_cols = [c for c in college_cols if c in college_df.columns]

college_df_clean = college_df[college_cols]

print("Selected columns:", len(college_cols))
print(college_df_clean.shape)

Selected columns: 36
(4571, 36)


In [12]:
missing_pct = college_df_clean.isna().mean().sort_values(ascending=False)

print("Top missing columns:")
print(missing_pct.head(15))

Top missing columns:
qb_Att               0.942682
qb_Cmp%              0.942682
qb_Yds               0.942682
qb_Cmp               0.942682
qb_Y/G               0.942682
qb_AY/A              0.942682
qb_Y/A               0.942682
qb_Int               0.942682
qb_TD                0.942682
rb_rec_Y_per_Rec     0.896303
rb_rush_Y_per_Att    0.895428
rb_rush_Att          0.895428
rb_rec_TD            0.895428
rb_rec_Yds           0.895428
rb_rec_Rec           0.895428
dtype: float64


In [13]:
# Ensure uniqueness
assert college_df_clean.duplicated(subset=["draft_year", "pick"]).sum() == 0

print("✅ No duplicate (year, pick) pairs")

# Check ranges
print(college_df_clean.describe())

✅ No duplicate (year, pick) pairs
        draft_year         pick        round            G          age  \
count  4571.000000  4571.000000  4571.000000  4531.000000  4566.000000   
mean   2011.987093   120.129075     3.971997    35.548665    22.444809   
std       6.439013    73.734772     2.004394    12.206447     0.915916   
min    2000.000000     1.000000     1.000000     1.000000    20.000000   
25%    2007.000000    56.000000     2.000000    26.000000    22.000000   
50%    2012.000000   115.000000     4.000000    37.000000    22.000000   
75%    2018.000000   184.000000     6.000000    46.000000    23.000000   
max    2022.000000   262.000000     7.000000    66.000000    29.000000   

       avg_team_sp       qb_Cmp       qb_Att     qb_Cmp%        qb_Yds  ...  \
count  2489.000000   262.000000   262.000000  262.000000    262.000000  ...   
mean     10.480089   612.438931   999.385496   60.129389   7753.942748  ...   
std      11.922393   269.641372   420.183740    6.966659   321

In [14]:
college_df_clean.to_csv(
    PROCESSED_DIR / "college_clean_for_merge.csv",
    index=False
)

In [16]:
# Prepare combine data
combine_av_df["draft_year"] = combine_av_df["draft_year"].astype(int)
combine_av_df["pick"] = combine_av_df["pick"].astype(int)

# Merge
merged_df = combine_av_df.merge(
    college_df_clean,
    on=["draft_year", "pick"],
    how="left"
)

print("Merged shape:", merged_df.shape)

Merged shape: (6644, 60)


In [18]:
merged_df.columns

Index(['draft_year', 'round_x', 'pick', 'team', 'hof', 'position', 'category',
       'side', 'college', 'age_x', 'to', 'allpro', 'probowls',
       'seasons_started', 'dr_av', 'player', 'pos_x', 'school', 'ht', 'wt',
       'forty', 'bench', 'vertical', 'broad_jump', 'cone', 'shuttle',
       'round_y', 'pos_y', 'G', 'age_y', 'avg_team_sp', 'qb_Cmp', 'qb_Att',
       'qb_Cmp%', 'qb_Yds', 'qb_TD', 'qb_Int', 'qb_Y/A', 'qb_AY/A', 'qb_Y/G',
       'rb_rush_Att', 'rb_rush_Yds', 'rb_rush_Y_per_Att', 'rb_rush_TD',
       'rb_rec_Rec', 'rb_rec_Yds', 'rb_rec_Y_per_Rec', 'rb_rec_TD', 'rec_Rec',
       'rec_Yds', 'rec_Y/R', 'rec_TD', 'rec_Y/G', 'Sk', 'Solo', 'Ast', 'Comb',
       'TFL', 'Int', 'PD'],
      dtype='str')

In [19]:
# Compare round
print("Round mismatch:",
      (merged_df["round_x"] != merged_df["round_y"]).sum())

# Compare position
print("Pos mismatch:",
      (merged_df["pos_x"] != merged_df["pos_y"]).sum())

# Compare age
print("Age mismatch:",
      (merged_df["age_x"] != merged_df["age_y"]).sum())

Round mismatch: 3342
Pos mismatch: 4893
Age mismatch: 2078


In [20]:
print("Round mismatch (non-null only):",
      ((merged_df["round_x"] != merged_df["round_y"]) &
       merged_df["round_y"].notna()).sum())

print("Pos mismatch (non-null only):",
      ((merged_df["pos_x"] != merged_df["pos_y"]) &
       merged_df["pos_y"].notna()).sum())

print("Age mismatch (non-null only):",
      ((merged_df["age_x"] != merged_df["age_y"]) &
       merged_df["age_y"].notna()).sum())

Round mismatch (non-null only): 1269
Pos mismatch (non-null only): 2820
Age mismatch (non-null only): 0


In [21]:
merged_df = merged_df.rename(columns={
    "round_x": "round",
    "pos_x": "pos_nfl",     # from combine
    "pos_y": "pos_college", # from college
    "age_x": "age"
})

In [22]:
merged_df = merged_df.drop(columns=["round_y", "age_y"])

In [ ]:
pos_diff = (
    (merged_df["pos_nfl"] != merged_df["pos_college"]) &
    merged_df["pos_college"].notna()
).mean()

print("Position disagreement rate:", pos_diff) # College production vs Combine have different position naming systems

Position disagreement rate: 0.4244431065623119


In [24]:
# =========================
# FINAL SANITY CHECKS
# =========================

print("=== BASIC INFO ===")
print("Shape:", merged_df.shape)
print("Columns:", len(merged_df.columns))


# -------------------------
# 1. Key integrity
# -------------------------
print("\n=== KEY CHECKS ===")

dup_keys = merged_df.duplicated(subset=["draft_year", "pick"]).sum()
print("Duplicate (year, pick):", dup_keys)

assert dup_keys == 0, "❌ Duplicate (draft_year, pick) found!"

print("✅ Keys are unique")


# -------------------------
# 2. Merge quality
# -------------------------
print("\n=== MERGE QUALITY ===")

if "avg_team_sp" in merged_df.columns:
    match_rate = merged_df["avg_team_sp"].notna().mean()
    print(f"Match rate (college features): {match_rate:.3f}")

    print("Matched rows:", merged_df["avg_team_sp"].notna().sum())
    print("Unmatched rows:", merged_df["avg_team_sp"].isna().sum())


# -------------------------
# 3. Missing values
# -------------------------
print("\n=== MISSING VALUES (TOP 15) ===")

missing_pct = merged_df.isna().mean().sort_values(ascending=False)
print(missing_pct.head(15))


# -------------------------
# 4. Target sanity (dr_av)
# -------------------------
print("\n=== TARGET CHECK (dr_av) ===")

print("dr_av missing:", merged_df["dr_av"].isna().sum())
print(merged_df["dr_av"].describe())


# -------------------------
# 5. Draft distribution
# -------------------------
print("\n=== DRAFT DISTRIBUTION ===")

print("Years:", merged_df["draft_year"].min(), "→", merged_df["draft_year"].max())
print("Pick range:", merged_df["pick"].min(), "→", merged_df["pick"].max())

if "round" in merged_df.columns:
    print("Rounds:", sorted(merged_df["round"].dropna().unique()))


# -------------------------
# 6. Position sanity
# -------------------------
print("\n=== POSITION CHECK ===")

if "pos_nfl" in merged_df.columns:
    print("Top NFL positions:")
    print(merged_df["pos_nfl"].value_counts().head())

if "pos_college" in merged_df.columns:
    print("\nTop College positions:")
    print(merged_df["pos_college"].value_counts().head())


# -------------------------
# 7. Numeric sanity
# -------------------------
print("\n=== NUMERIC SUMMARY ===")

print(merged_df.describe())


print("\n✅ SANITY CHECKS COMPLETE")

=== BASIC INFO ===
Shape: (6644, 58)
Columns: 58

=== KEY CHECKS ===
Duplicate (year, pick): 0
✅ Keys are unique

=== MERGE QUALITY ===
Match rate (college features): 0.375
Matched rows: 2489
Unmatched rows: 4155

=== MISSING VALUES (TOP 15) ===
qb_Yds              0.960566
qb_Y/G              0.960566
qb_Int              0.960566
qb_Y/A              0.960566
qb_AY/A             0.960566
qb_TD               0.960566
qb_Att              0.960566
qb_Cmp%             0.960566
qb_Cmp              0.960566
rb_rec_Y_per_Rec    0.928657
rb_rush_Yds         0.928055
rb_rec_Rec          0.928055
rb_rec_Yds          0.928055
rb_rush_TD          0.928055
rb_rush_Att         0.928055
dtype: float64

=== TARGET CHECK (dr_av) ===
dr_av missing: 993
count    5651.000000
mean       12.983012
std        17.453211
min        -4.000000
25%         2.000000
50%         7.000000
75%        17.000000
max       168.000000
Name: dr_av, dtype: float64

=== DRAFT DISTRIBUTION ===
Years: 2000 → 2025
Pick range: 

In [25]:
merged_df = merged_df.dropna(subset=["dr_av"])

In [26]:
merged_df["round"] = ((merged_df["pick"] - 1) // 32) + 1

In [27]:
print(sorted(merged_df["round"].unique()))

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [28]:
def get_round(pick):
    if pick <= 32: return 1
    elif pick <= 64: return 2
    elif pick <= 96: return 3
    elif pick <= 128: return 4
    elif pick <= 160: return 5
    elif pick <= 192: return 6
    else: return 7

merged_df["round"] = merged_df["pick"].apply(get_round)

In [29]:
print(sorted(merged_df["round"].unique()))

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [30]:
# =========================
# SAVE CLEAN DATASET
# =========================

output_path = PROCESSED_DIR / "final_merged_player_data.csv"

merged_df.to_csv(output_path, index=False)

print("✅ Saved to:", output_path)

✅ Saved to: ..\data\processed\final_merged_player_data.csv
